In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"]="false"
from functools import partial
import time
import glob
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
import jax
import jax.numpy as jnp
# jax.config.update("jax_enable_x64", True)
gpus = jax.devices()
print(gpus)

jax.config.update("jax_default_device", gpus[0])

import diffrax
import equinox as eqx
import optax

from haiku import PRNGSequence

import exciting_environments as excenvs

from dmpe.data_management import DataPaths
from dmpe.models.models import NeuralEulerODECartpole
from dmpe.evaluation.experiment_utils import get_experiment_ids, load_experiment_results, load_all_experiment_results
from dmpe.evaluation.data_evaluation import DataEvaluator, JensenShannonDivergence
from dmpe.evaluation.model_evaluation import ModelWrapper, ModelEvaluator, PredictionComparison, NodeModelWrapper, EnvWrapper
from dmpe.evaluation.utils import default_constraint_function

from dmpe.utils.env_utils.cart_pole_utils import setup_env as setup_cart_pole_env

In [ ]:
from dmpe.models.model_training import ModelTrainer
from dmpe.evaluation.exp_data_model_learning import train_model_on_experiment_data, ModelExpDataResult

In [ ]:
from plot_helpers import plot_jsd_model_relation

In [ ]:
import matplotlib as mpl
from matplotlib import rc
rc('font',**{'family':'serif','serif':['Helvetica']})
mpl.rcParams['text.usetex'] = True
mpl.rcParams.update({'font.size': 10})
mpl.rcParams['text.latex.preamble']=r"\usepackage{bm}\usepackage{amsmath}\usepackage{upgreek}"

## Eval plots

In [ ]:
plot_jsd_model_relation(
    data_path=DataPaths().cs_experiments / "model_on_exp_data" / "various_exp_together_out",
    model_class=NeuralEulerODECartpole,
    verbose=False,
)

In [ ]:
fig, ax = plt.subplots(1,1, figsize=(6,6))
colors = plt.rcParams["axes.prop_cycle"]()

for data_length in [1000, 2000, 3000, 4000]:
    data_path = DataPaths().cs_experiments / "model_on_exp_data" / "various_exp_together_out" / f"{data_length}_len"

    means = []
    medians = []
    jsds = []
    
    n_results = len(glob.glob(str(data_path / "*.eqx")))
    print("# or results:", n_results)

    for idx in range(n_results):
        result = ModelExpDataResult.from_file(
            filename= data_path / f"test_{idx}.eqx",
            model_class=NeuralEulerODECartpole,
        )
        means.append(jnp.mean(jnp.array(result.model_errors), axis=0)[-1])
        medians.append(jnp.median(jnp.array(result.model_errors), axis=0)[-1])
        jsds.append(result.data_jsd)


    
    ax.scatter(jsds, medians, s=25, marker="x", c=next(colors)["color"], label=f"{data_length} data points")

# ax.set_ylabel(r"$\\mathcal{L}_{\\mathcal{M}}$")
# ax.set_xlabel(r"$\\mathcal{L}_{\mathrm{JSD}}$")

ax.set_ylabel(r"$\mathcal{L}_{\mathcal{M}}$")
ax.set_xlabel(r"$\mathcal{L}_{\mathrm{JSD}}$")
ax.grid(True)
ax.set_yscale('log')
ax.legend()
plt.show()

## Train model from scratch:

In [ ]:
# env, _ = setup_cart_pole_env()
# wrapped_env = EnvWrapper(env)

# points_per_dim = 20

# model_evaluator = ModelEvaluator(
#     constraint_function=default_constraint_function,
#     gt_model=wrapped_env,
#     obs_dim=4,
#     act_dim=1,
#     validation_points_per_dim=points_per_dim,
#     tau=env.tau,
# )

# data_evaluator = DataEvaluator(
#     constraint_function=default_constraint_function,
#     data_dim=5,
#     points_per_dim=points_per_dim,
# )

In [ ]:
# lr = 1e-4
# n_iters = 100
# n_datapoints = 1_000

# def featurize_theta_cart_pole(obs):
#     """The angle itself is difficult to properly interpret in the loss as angles
#     such as 1.99 * pi and 0 are essentially the same. Therefore the angle is
#     transformed to sin(phi) and cos(phi) for comparison in the loss."""
#     feat_obs = jnp.stack(
#         [obs[..., 0], obs[..., 1], jnp.sin(obs[..., 2] * jnp.pi), jnp.cos(obs[..., 2] * jnp.pi), obs[..., 3]],
#         axis=-1,
#     )
#     return feat_obs


# cart_pole_data_path = DataPaths().cs_experiments / "dmpe" / "cart_pole" / "old_results" 
# dmpe_experiment_results = load_all_experiment_results(cart_pole_data_path, model_class=NeuralEulerODECartpole)

In [ ]:
# for experiment_idx, dmpe_experiment_result in enumerate(dmpe_experiment_results):

#     exp_id = dmpe_experiment_result["exp_id"]

#     print(exp_id)
#     observations = dmpe_experiment_result["observations"][:n_datapoints]
#     actions = dmpe_experiment_result["actions"][:n_datapoints]
#     internal_model = dmpe_experiment_result["model"]
    
#     list_trained_models = []
#     list_model_errors_dmpe = []
    
#     seeds = [1,2,3,4,5,6,7]
    
#     for seed_idx, seed in enumerate(seeds):

#         model_class=NeuralEulerODECartpole
#         model_params=dict(
#             obs_dim=env.reset(env.env_properties)[0].shape[0],
#             action_dim=env.action_dim,
#             width_size=64,
#             depth=2,
#         )
        
#         trained_model, model_errors_dmpe = train_model_on_experiment_data(
#             key=jax.random.key(seed),
#             observations=observations,
#             actions=actions,
#             model_trainer_params=dict(
#                 start_learning=None,
#                 training_batch_size=128,
#                 n_train_steps=1_000,
#                 sequence_length=10,
#                 featurize=featurize_theta_cart_pole,
#                 model_optimizer=optax.adabelief(lr),
#                 tau=env.tau,
#             ),
#             model_params=model_params,
#             n_iters=n_iters,
#             model_class=model_class,
#             model_evaluator=model_evaluator,
#         )
    
#         list_trained_models.append(trained_model)
#         list_model_errors_dmpe.append(model_errors_dmpe)

#     result = ModelExpDataResult.from_data(
#         exp_id=exp_id,
#         seeds=seeds,
#         observations=observations,
#         actions=actions,
#         data_jsd=data_evaluator.get_metrics(data_points = jnp.concatenate([observations, actions], axis=-1))["jsd"],
#         model_params=model_params,
#         model_class=model_class,
#         models=list_trained_models,
#         model_errors=jnp.array(list_model_errors_dmpe),
#     )

#     save_folder = DataPaths().cs_experiments / "model_on_exp_data"
#     file_path = save_folder / f"test_{experiment_idx}.eqx"
#     result.save_to_file(file_path)
#     print(20 * "#")

In [ ]:
results = [
    ModelExpDataResult.from_file(
        filename=save_folder / f"test_{idx}.eqx",
        model_class=NeuralEulerODECartpole,
    ) for idx in range(4)
]
results

In [ ]:
for result in results:
    fig, axs = result.visualize()
    plt.show()
    print("")

---

In [ ]:
for result in results:
    for model_errors_dmpe in result.model_errors:
        if len(model_errors_dmpe) == 1:
            plt.hlines(model_errors_dmpe[0], xmin=0.0, xmax=len(model_errors_dmpe), color="b")
        else:
            plt.plot(model_errors_dmpe)

    plt.grid(True)
    plt.yscale("log")
    
    plt.show()
    
    ###
    
    colors = plt.rcParams["axes.prop_cycle"]()
    c1 = next(colors)["color"]
    c2 = next(colors)["color"]
    c3 = next(colors)["color"]
    mean = jnp.nanmean(jnp.array(result.model_errors), axis=0)
    std = jnp.nanstd(jnp.array(result.model_errors), axis=0)
    
    plt.plot(
        jnp.arange(0, len(mean), 1),
        mean,
        color=c1,
    )
    plt.fill_between(
        jnp.arange(0, len(mean), 1),
        mean - std,
        mean + std,
        color=c1,
        alpha=0.1,
    )
    
    # plt.hlines(metric, xmin=0.0, xmax=len(result.model_errors), color=c3)
    plt.grid(True)
    plt.yscale("log")
    plt.show()

In [ ]:
means = []
medians = []
jsds = []

for result in results:
    means.append(jnp.mean(jnp.array(result.model_errors), axis=0)[-1])
    medians.append(jnp.median(jnp.array(result.model_errors), axis=0)[-1])
    jsds.append(result.data_jsd)

    #plt.scatter(means, jsds)
plt.scatter(jsds, medians)

In [ ]:
from dmpe.evaluation.exp_data_model_learning import ModelExpDataResult

In [ ]:
save_folder = DataPaths().cs_experiments / "model_on_exp_data"
file_path = save_folder / "test.eqx"

result.save_to_file(file_path)

In [ ]:
result_loaded = ModelExpDataResult.from_file(file_path, model_class=NeuralEulerODECartpole)

In [ ]:
[a for a in dir(result_loaded) if not (a.startswith('__') or a.startswith('_'))]

In [ ]:
print(jnp.all(result_loaded.actions == result.actions))
print(jnp.all(result_loaded.data_jsd == result.data_jsd))
print(result_loaded.exp_id == result.exp_id)
print(result_loaded.model_class == result.model_class)
print(result_loaded.model_params == result.model_params)
print(result_loaded.models == result.models)
print(result_loaded.n_actions == result.n_actions)
print(result_loaded.n_datapoints == result.n_datapoints)
print(result_loaded.n_iters == result.n_iters)
print(result_loaded.n_obs == result.n_obs)
print(jnp.all(result_loaded.observations == result.observations))
print(result_loaded.seeds == result.seeds)

In [ ]:
_, metric = model_evaluator.default_metrics["pred_comp"](NodeModelWrapper(internal_model), model_evaluator.gt_model)

for model_errors_dmpe in list_model_errors_dmpe:
    if len(model_errors_dmpe) == 1:
        plt.hlines(model_errors_dmpe[0], xmin=0.0, xmax=len(model_errors_dmpe), color="b")
    else:
        plt.plot(model_errors_dmpe)

plt.hlines(metric, xmin=0.0, xmax=len(model_errors_dmpe), color="r")
plt.grid(True)
plt.yscale("log")

plt.show()

###

colors = plt.rcParams["axes.prop_cycle"]()
c1 = next(colors)["color"]
c2 = next(colors)["color"]
c3 = next(colors)["color"]
mean = jnp.nanmean(jnp.array(list_model_errors_dmpe), axis=0)
std = jnp.nanstd(jnp.array(list_model_errors_dmpe), axis=0)

plt.plot(
    jnp.arange(0, len(mean), 1),
    mean,
    color=c1,
)
plt.fill_between(
    jnp.arange(0, len(mean), 1),
    mean - std,
    mean + std,
    color=c1,
    alpha=0.1,
)

plt.hlines(metric, xmin=0.0, xmax=len(model_errors_dmpe), color=c3)
plt.grid(True)
plt.yscale("log")
plt.show()

In [ ]:
mean.shape

In [ ]:
result = ModelExpDataResult

In [ ]:
model_params=dict(
    obs_dim=env.reset(env.env_properties)[0].shape[0],
    action_dim=env.action_dim,
    width_size=64,
    depth=2,
    key=None,
)

print(model_params["key"] is None)

if model_params["key"] is None:
    model_params["key"] = jax.random.PRNGKey(152)

print(model_params["key"] is None)
    

In [ ]:
igoats_experiment_results = load_all_experiment_results(DataPaths().cs_experiments / "igoats" / "cart_pole", model_class=None)

igoats_observations = igoats_experiment_results[0]["observations"][:n_datapoints]
igoats_actions = igoats_experiment_results[0]["actions"][:n_datapoints]
internal_model = None

print("data_jsd:", data_evaluator.get_metrics(data_points = jnp.concatenate([igoats_observations, igoats_actions], axis=-1))["jsd"])

trained_model_igoats, model_errors_igoats = train_cartpole_model(
    key=jax.random.key(4444),
    observations=igoats_observations,
    actions=igoats_actions,
    n_iters=n_iters,
    lr=lr,
)
plt.plot(model_errors_igoats)
plt.hlines(metric, xmin=0.0, xmax=len(model_errors_dmpe), color="r")
plt.yscale("log")

In [ ]:
plt.plot(model_errors_dmpe, label="dmpe")
plt.plot(model_errors_igoats, label="igoats")
plt.hlines(metric, xmin=0.0, xmax=len(model_errors_dmpe), color="r")
plt.yscale("log")
plt.legend()

In [ ]:
 _, metric_dmpe = model_evaluator.default_metrics["pred_comp"](NodeModelWrapper(trained_model), model_evaluator.gt_model)
 _, metric_igoats = model_evaluator.default_metrics["pred_comp"](NodeModelWrapper(trained_model_igoats), model_evaluator.gt_model)

In [ ]:
plt.hlines(metric_dmpe, xmin=0.0, xmax=100, color="orange", label="dmpe")
plt.hlines(metric_igoats, xmin=0.0, xmax=100, color="b", label="goats")
plt.hlines(metric, xmin=0.0, xmax=100, color="r", label="internal")
plt.yscale("log")
plt.legend()

- n_train_steps in the internal jax for loop is like 10 times faster than a python for loop for 2 step sequences.
- 2 step sequences are mega bad for the 2 step prediction performance?!
- you have to set up a big automated experiment for this!
- Is it possible to parallelize this somehow?
- Epoch-based better?